In [1]:
import os
import json
import torch
from pathlib import Path

# ── GPU setup ─────────────────────────────────────────────────────────────────
os.environ["CUDA_VISIBLE_DEVICES"] = "0,1"
print("CUDA devices:", os.environ["CUDA_VISIBLE_DEVICES"])
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)

# ── Reproducibility ────────────────────────────────────────────────────────────
SEED = 42
torch.manual_seed(SEED)

# ── Class counts ──────────────────────────────────────────────────────────────
SOURCE_NUM_CLASSES = 1 + 1   # 1 class + background
TARGET_NUM_CLASSES = 52 + 1  # 52 classes + background

# ── Fixed hyper-parameters ─────────────────────────────────────────────────────
FIXED = dict(
    epochs       = 30,
    patience     = 50,
    batch        = 16,
    imgsz        = 640,
    workers      = 2,
    optimizer    = "SGD",
    momentum     = 0.937,
    weight_decay = 0.0005,
    seed         = SEED,
)

# ── Per-stage learning rates ───────────────────────────────────────────────────
LR_SOURCE = 0.01     # Stage 1: fine-tune on source domain (resin)
LR_TARGET = 0.001    # Stage 2: fine-tune on target domain (pollen)

print("Setup complete.")
print(f"Fixed params: {json.dumps(FIXED, indent=2)}")

CUDA devices: 0,1
Using device: cuda
Setup complete.
Fixed params: {
  "epochs": 30,
  "patience": 50,
  "batch": 16,
  "imgsz": 640,
  "workers": 2,
  "optimizer": "SGD",
  "momentum": 0.937,
  "weight_decay": 0.0005,
  "seed": 42
}


In [2]:
from torchvision.models.detection import fasterrcnn_resnet50_fpn, FasterRCNN_ResNet50_FPN_Weights
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor

def build_model(num_classes, pretrained_coco=True):
    weights = FasterRCNN_ResNet50_FPN_Weights.DEFAULT if pretrained_coco else None
    model = fasterrcnn_resnet50_fpn(weights=weights)
    in_features = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)
    return model

def load_for_target(checkpoint_path, num_classes):
    model = build_model(num_classes=num_classes, pretrained_coco=False)
    checkpoint = torch.load(checkpoint_path, map_location=DEVICE)
    
    # Remove the head weights — they have different sizes between stages
    head_keys = [k for k in checkpoint if "box_predictor" in k]
    for k in head_keys:
        del checkpoint[k]
    
    # Load only backbone + FPN weights, skip the head
    model.load_state_dict(checkpoint, strict=False)
    return model

In [3]:
import os
import torch
import json
import time
import numpy as np
from pathlib import Path
from PIL import Image
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms.functional as TF
from torchvision.models.detection import fasterrcnn_resnet50_fpn, FasterRCNN_ResNet50_FPN_Weights
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor

# ── Dataset Class ──────────────────────────────────────────────────────────────
class PollenDataset(Dataset):
    """
    Expects YOLO-format annotations:
      images/  -> .jpg or .png files
      labels/  -> .txt files with same stem as image
    Each label line: class_id cx cy w h (normalized 0-1)
    """
    def __init__(self, img_dir, label_dir, class_names, transforms=None):
        self.img_dir     = Path(img_dir)
        self.label_dir   = Path(label_dir)
        self.class_names = class_names
        self.transforms  = transforms

        # Only keep images that have a corresponding label file
        self.images = sorted([
            p for p in self.img_dir.glob("*")
            if p.suffix.lower() in (".jpg", ".jpeg", ".png")
            and (self.label_dir / (p.stem + ".txt")).exists()
        ])

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img_path   = self.images[idx]
        label_path = self.label_dir / (img_path.stem + ".txt")

        # Load image
        img = Image.open(img_path).convert("RGB")
        w, h = img.size

        # Parse YOLO labels → convert to absolute xyxy
        boxes, labels = [], []
        with open(label_path) as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) != 5:
                    continue
                cls, cx, cy, bw, bh = map(float, parts)
                # YOLO normalized → absolute pixel coords
                x1 = (cx - bw / 2) * w
                y1 = (cy - bh / 2) * h
                x2 = (cx + bw / 2) * w
                y2 = (cy + bh / 2) * h
                boxes.append([x1, y1, x2, y2])
                labels.append(int(cls) + 1)  # +1 because 0 = background

        # Handle images with no valid boxes
        if len(boxes) == 0:
            boxes  = torch.zeros((0, 4), dtype=torch.float32)
            labels = torch.zeros((0,),   dtype=torch.int64)
        else:
            boxes  = torch.tensor(boxes,  dtype=torch.float32)
            labels = torch.tensor(labels, dtype=torch.int64)

        target = {
            "boxes":    boxes,
            "labels":   labels,
            "image_id": torch.tensor([idx]),
        }

        img = TF.to_tensor(img)  # → [C, H, W], float32 in [0, 1]

        if self.transforms:
            img, target = self.transforms(img, target)

        return img, target


# ── Collate function (required for variable-size targets) ─────────────────────
def collate_fn(batch):
    return tuple(zip(*batch))


# ── Early stopping ─────────────────────────────────────────────────────────────
class EarlyStopping:
    def __init__(self, patience=50, min_delta=0.0):
        self.patience   = patience
        self.min_delta  = min_delta
        self.best_loss  = float("inf")
        self.counter    = 0
        self.should_stop = False

    def step(self, val_loss):
        if val_loss < self.best_loss - self.min_delta:
            self.best_loss = val_loss
            self.counter   = 0
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.should_stop = True


# ── Training loop ──────────────────────────────────────────────────────────────
def train_one_epoch(model, optimizer, loader, device):
    model.train()
    total_loss = 0.0
    for images, targets in loader:
        images  = [img.to(device) for img in images]
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

        loss_dict = model(images, targets)
        loss      = sum(loss_dict.values())

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
    return total_loss / len(loader)


def evaluate(model, loader, device):
    """Compute average validation loss."""
    model.train()  # Faster R-CNN needs train() mode to return losses
    total_loss = 0.0
    with torch.no_grad():
        for images, targets in loader:
            images  = [img.to(device) for img in images]
            targets = [{k: v.to(device) for k, v in t.items()} for t in targets]
            loss_dict  = model(images, targets)
            total_loss += sum(loss_dict.values()).item()
    return total_loss / len(loader)


def train(model, train_loader, val_loader, optimizer, scheduler,
          epochs, patience, device, checkpoint_path):

    early_stopping = EarlyStopping(patience=patience)
    best_val_loss  = float("inf")
    history        = {"train_loss": [], "val_loss": []}

    for epoch in range(1, epochs + 1):
        t0         = time.time()
        train_loss = train_one_epoch(model, optimizer, train_loader, device)
        val_loss   = evaluate(model, val_loader, device)
        scheduler.step()

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)

        elapsed = time.time() - t0
        print(f"Epoch {epoch:03d}/{epochs} | "
              f"Train Loss: {train_loss:.4f} | "
              f"Val Loss: {val_loss:.4f} | "
              f"Time: {elapsed:.1f}s")

        # Save best checkpoint
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save(model.state_dict(), checkpoint_path)
            print(f"  ✓ Saved best model → {checkpoint_path}")

        early_stopping.step(val_loss)
        if early_stopping.should_stop:
            print(f"Early stopping at epoch {epoch}.")
            break

    return history

In [4]:
SOURCE_CLASSES = ["resin"]

RESIN_TRAIN = "/storage/praha1/home/mamedove/00_Data/2025_Pollen/Single_class/big-set"
RESIN_VAL   = "/storage/praha1/home/mamedove/00_Data/2025_Pollen/Single_class/small-set"

print("=== Stage 1: Resin (source domain) ===")

train_dataset_s = PollenDataset(RESIN_TRAIN, RESIN_TRAIN, SOURCE_CLASSES)
val_dataset_s   = PollenDataset(RESIN_VAL,   RESIN_VAL,   SOURCE_CLASSES)

print(f"Train images: {len(train_dataset_s)}")
print(f"Val images:   {len(val_dataset_s)}")

train_loader_s = DataLoader(train_dataset_s, batch_size=FIXED["batch"],
                            shuffle=True,  collate_fn=collate_fn,
                            num_workers=FIXED["workers"])
val_loader_s   = DataLoader(val_dataset_s,   batch_size=FIXED["batch"],
                            shuffle=False, collate_fn=collate_fn,
                            num_workers=FIXED["workers"])

model_s = build_model(num_classes=SOURCE_NUM_CLASSES, pretrained_coco=True)
model_s.to(DEVICE)

optimizer_s = torch.optim.SGD(model_s.parameters(), lr=LR_SOURCE,
                              momentum=FIXED["momentum"],
                              weight_decay=FIXED["weight_decay"])
scheduler_s = torch.optim.lr_scheduler.CosineAnnealingLR(
                optimizer_s, T_max=FIXED["epochs"], eta_min=LR_SOURCE * 0.01)

history_s = train(model_s, train_loader_s, val_loader_s,
                  optimizer_s, scheduler_s,
                  epochs=FIXED["epochs"], patience=FIXED["patience"],
                  device=DEVICE, checkpoint_path="resin_best.pth")

=== Stage 1: Resin (source domain) ===
Train images: 230
Val images:   46
Epoch 001/30 | Train Loss: 0.5787 | Val Loss: 0.3927 | Time: 32.3s
  ✓ Saved best model → resin_best.pth
Epoch 002/30 | Train Loss: 0.2442 | Val Loss: 0.2503 | Time: 20.3s
  ✓ Saved best model → resin_best.pth
Epoch 003/30 | Train Loss: 0.1875 | Val Loss: 0.3025 | Time: 20.5s
Epoch 004/30 | Train Loss: 0.1534 | Val Loss: 0.2912 | Time: 20.5s
Epoch 005/30 | Train Loss: 0.1386 | Val Loss: 0.3002 | Time: 20.5s


KeyboardInterrupt: 

In [ ]:
class PollenDatasetFromTxt(Dataset):
    def __init__(self, txt_path, class_names, transforms=None, imgsz=640):
        self.imgsz = imgsz
        self.class_names = class_names
        self.transforms  = transforms

        txt_path = Path(txt_path)
        base_dir = txt_path.parent

        with open(txt_path) as f:
            all_paths = [base_dir / l.strip() for l in f if l.strip()]

        # Label is in the SAME folder as the image
        self.images = []
        for img_path in all_paths:
            img_path   = img_path.resolve()
            label_path = img_path.with_suffix(".txt")  # same folder, same stem
            if img_path.exists() and label_path.exists():
                self.images.append(img_path)

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img_path   = self.images[idx]
        label_path = img_path.with_suffix(".txt")  # same folder, same stem

        img = Image.open(img_path).convert("RGB")
        w, h = img.size

        boxes, labels = [], []
        with open(label_path) as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) != 5:
                    continue
                cls, cx, cy, bw, bh = map(float, parts)
                x1 = (cx - bw / 2) * w
                y1 = (cy - bh / 2) * h
                x2 = (cx + bw / 2) * w
                y2 = (cy + bh / 2) * h
                boxes.append([x1, y1, x2, y2])
                labels.append(int(cls) + 1)

        if len(boxes) == 0:
            boxes  = torch.zeros((0, 4), dtype=torch.float32)
            labels = torch.zeros((0,),   dtype=torch.int64)
        else:
            boxes  = torch.tensor(boxes,  dtype=torch.float32)
            labels = torch.tensor(labels, dtype=torch.int64)

        target = {
            "boxes":    boxes,
            "labels":   labels,
            "image_id": torch.tensor([idx]),
        }

        img = TF.to_tensor(img)
        
        # Resize image and scale boxes accordingly
        orig_h, orig_w = img.shape[1], img.shape[2]
        img = TF.resize(img, [self.imgsz, self.imgsz])
        
        # Scale boxes to new size
        if len(target["boxes"]) > 0:
            scale_x = self.imgsz / orig_w
            scale_y = self.imgsz / orig_h
            target["boxes"][:, [0, 2]] *= scale_x
            target["boxes"][:, [1, 3]] *= scale_y

        return img, target

In [ ]:
POLLEN_TRAIN_TXT = "/storage/praha1/home/mamedove/00_Data/2025_Pollen/Multi_class/multi-class-set/train.txt"
POLLEN_VAL_TXT   = "/storage/praha1/home/mamedove/00_Data/2025_Pollen/Multi_class/multi-class-set/val.txt"
TARGET_CLASSES   = [str(i) for i in range(1, 53)]

train_dataset_t = PollenDatasetFromTxt(POLLEN_TRAIN_TXT, TARGET_CLASSES, imgsz=640)
val_dataset_t   = PollenDatasetFromTxt(POLLEN_VAL_TXT,   TARGET_CLASSES, imgsz=640)

print(f"Train images: {len(train_dataset_t)}")
print(f"Val images:   {len(val_dataset_t)}")

print("=== Stage 2: Pollen (target domain) ===")


train_loader_t = DataLoader(train_dataset_t, batch_size=FIXED["batch"],
                            shuffle=True,  collate_fn=collate_fn,
                            num_workers=FIXED["workers"])
val_loader_t   = DataLoader(val_dataset_t,   batch_size=FIXED["batch"],
                            shuffle=False, collate_fn=collate_fn,
                            num_workers=FIXED["workers"])

model_t = load_for_target("resin_best.pth", TARGET_NUM_CLASSES)
model_t.to(DEVICE)

optimizer_t = torch.optim.SGD(model_t.parameters(), lr=LR_TARGET,
                              momentum=FIXED["momentum"],
                              weight_decay=FIXED["weight_decay"])
scheduler_t = torch.optim.lr_scheduler.CosineAnnealingLR(
                optimizer_t, T_max=FIXED["epochs"], eta_min=LR_TARGET * 0.01)

history_t = train(model_t, train_loader_t, val_loader_t,
                  optimizer_t, scheduler_t,
                  epochs=FIXED["epochs"], patience=FIXED["patience"],
                  device=DEVICE, checkpoint_path="pollen_best.pth")

print(f"Best resin val loss:  {min(history_s['val_loss']):.4f}")
print(f"Best pollen val loss: {min(history_t['val_loss']):.4f}")